In [4]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

spark = SparkSession.builder.appName("EmployeeSalaryAnalytics").getOrCreate()

employee_data = [
(101, "Sravan", "Data Engineer", "IT", 75000, "Hyderabad", 28, "2021-05-10", "Male"),
(102, "Ravi", "Software Engineer", "IT", 68000, "Bangalore", 30, "2020-03-15", "Male"),
(103, "Priya", "Data Analyst", "Analytics", 62000, "Chennai", 26, "2022-01-12", "Female"),
(104, "Kiran", "Manager", "HR", 90000, "Mumbai", 35, "2018-07-19", "Male"),
(105, "Anjali", "HR Executive", "HR", 45000, "Pune", 24, "2023-02-20", "Female"),
(106, "Vikram", "Data Scientist", "Analytics", 98000, "Delhi", 32, "2019-11-25", "Male"),
(107, "Sneha", "Developer", "IT", 71000, "Hyderabad", 27, "2021-08-17", "Female"),
(108, "Rahul", "Tester", "QA", 55000, "Chennai", 29, "2020-06-10", "Male"),
(109, "Meena", "QA Lead", "QA", 83000, "Bangalore", 33, "2017-09-14", "Female"),
(110, "Arjun", "Support Engineer", "Support", 50000, "Pune", 31, "2022-04-11", "Male")
]

columns = [
    "emp_id",
    "name",
    "designation",
    "department",
    "salary",
    "city",
    "age",
    "joining_date",
    "gender"
]

emp_df = spark.createDataFrame(employee_data, columns)

emp_df.show()

+------+------+-----------------+----------+------+---------+---+------------+------+
|emp_id|  name|      designation|department|salary|     city|age|joining_date|gender|
+------+------+-----------------+----------+------+---------+---+------------+------+
|   101|Sravan|    Data Engineer|        IT| 75000|Hyderabad| 28|  2021-05-10|  Male|
|   102|  Ravi|Software Engineer|        IT| 68000|Bangalore| 30|  2020-03-15|  Male|
|   103| Priya|     Data Analyst| Analytics| 62000|  Chennai| 26|  2022-01-12|Female|
|   104| Kiran|          Manager|        HR| 90000|   Mumbai| 35|  2018-07-19|  Male|
|   105|Anjali|     HR Executive|        HR| 45000|     Pune| 24|  2023-02-20|Female|
|   106|Vikram|   Data Scientist| Analytics| 98000|    Delhi| 32|  2019-11-25|  Male|
|   107| Sneha|        Developer|        IT| 71000|Hyderabad| 27|  2021-08-17|Female|
|   108| Rahul|           Tester|        QA| 55000|  Chennai| 29|  2020-06-10|  Male|
|   109| Meena|          QA Lead|        QA| 83000|Ban

Find Top 3 Highest Salaries Department-Wise

In [5]:
from pyspark.sql.window import Window
from pyspark.sql.functions import *

window_spec = Window.partitionBy("department") \
                    .orderBy(col("salary").desc())

top3_df = emp_df.withColumn(
    "rank",
    row_number().over(window_spec)
).filter(col("rank") <= 3)

top3_df.show()

+------+------+-----------------+----------+------+---------+---+------------+------+----+
|emp_id|  name|      designation|department|salary|     city|age|joining_date|gender|rank|
+------+------+-----------------+----------+------+---------+---+------------+------+----+
|   106|Vikram|   Data Scientist| Analytics| 98000|    Delhi| 32|  2019-11-25|  Male|   1|
|   103| Priya|     Data Analyst| Analytics| 62000|  Chennai| 26|  2022-01-12|Female|   2|
|   104| Kiran|          Manager|        HR| 90000|   Mumbai| 35|  2018-07-19|  Male|   1|
|   105|Anjali|     HR Executive|        HR| 45000|     Pune| 24|  2023-02-20|Female|   2|
|   101|Sravan|    Data Engineer|        IT| 75000|Hyderabad| 28|  2021-05-10|  Male|   1|
|   107| Sneha|        Developer|        IT| 71000|Hyderabad| 27|  2021-08-17|Female|   2|
|   102|  Ravi|Software Engineer|        IT| 68000|Bangalore| 30|  2020-03-15|  Male|   3|
|   109| Meena|          QA Lead|        QA| 83000|Bangalore| 33|  2017-09-14|Female|   1|

 Find Average Salary City-Wise

In [6]:
from pyspark.sql.functions import avg

city_avg_df = emp_df.groupBy("city") \
                    .agg(avg("salary").alias("average_salary"))

city_avg_df.show()

+---------+--------------+
|     city|average_salary|
+---------+--------------+
|Bangalore|       75500.0|
|  Chennai|       58500.0|
|   Mumbai|       90000.0|
|     Pune|       47500.0|
|Hyderabad|       73000.0|
|    Delhi|       98000.0|
+---------+--------------+



 Categorize Employees into Salary Bands

In [7]:
from pyspark.sql.functions import when

salary_band_df = emp_df.withColumn(
    "salary_band",
    when(col("salary") < 60000, "Low")
    .when(col("salary") <= 80000, "Medium")
    .otherwise("High")
)

salary_band_df.show()

+------+------+-----------------+----------+------+---------+---+------------+------+-----------+
|emp_id|  name|      designation|department|salary|     city|age|joining_date|gender|salary_band|
+------+------+-----------------+----------+------+---------+---+------------+------+-----------+
|   101|Sravan|    Data Engineer|        IT| 75000|Hyderabad| 28|  2021-05-10|  Male|     Medium|
|   102|  Ravi|Software Engineer|        IT| 68000|Bangalore| 30|  2020-03-15|  Male|     Medium|
|   103| Priya|     Data Analyst| Analytics| 62000|  Chennai| 26|  2022-01-12|Female|     Medium|
|   104| Kiran|          Manager|        HR| 90000|   Mumbai| 35|  2018-07-19|  Male|       High|
|   105|Anjali|     HR Executive|        HR| 45000|     Pune| 24|  2023-02-20|Female|        Low|
|   106|Vikram|   Data Scientist| Analytics| 98000|    Delhi| 32|  2019-11-25|  Male|       High|
|   107| Sneha|        Developer|        IT| 71000|Hyderabad| 27|  2021-08-17|Female|     Medium|
|   108| Rahul|     

Generate Yearly Salary Report

In [9]:
from pyspark.sql.functions import *

yearly_df = emp_df.withColumn(
    "year",
    year(to_date(col("joining_date")))
)

yearly_report = yearly_df.groupBy("year") \
    .agg(
        count("*").alias("employee_count"),
        avg("salary").alias("average_salary"),
        sum("salary").alias("total_salary")
    ) \
    .orderBy("year")

yearly_report.show()

+----+--------------+--------------+------------+
|year|employee_count|average_salary|total_salary|
+----+--------------+--------------+------------+
|2017|             1|       83000.0|       83000|
|2018|             1|       90000.0|       90000|
|2019|             1|       98000.0|       98000|
|2020|             2|       61500.0|      123000|
|2021|             2|       73000.0|      146000|
|2022|             2|       56000.0|      112000|
|2023|             1|       45000.0|       45000|
+----+--------------+--------------+------------+



Find Employees Earning Above Department Average

In [10]:
from pyspark.sql.functions import avg

dept_avg = emp_df.groupBy("department") \
    .agg(avg("salary").alias("dept_avg_salary"))

result_df = emp_df.join(
    dept_avg,
    on="department",
    how="inner"
).filter(
    col("salary") > col("dept_avg_salary")
)

result_df.show()

+----------+------+------+--------------+------+---------+---+------------+------+-----------------+
|department|emp_id|  name|   designation|salary|     city|age|joining_date|gender|  dept_avg_salary|
+----------+------+------+--------------+------+---------+---+------------+------+-----------------+
|        HR|   104| Kiran|       Manager| 90000|   Mumbai| 35|  2018-07-19|  Male|          67500.0|
|        IT|   101|Sravan| Data Engineer| 75000|Hyderabad| 28|  2021-05-10|  Male|71333.33333333333|
| Analytics|   106|Vikram|Data Scientist| 98000|    Delhi| 32|  2019-11-25|  Male|          80000.0|
|        QA|   109| Meena|       QA Lead| 83000|Bangalore| 33|  2017-09-14|Female|          69000.0|
+----------+------+------+--------------+------+---------+---+------------+------+-----------------+

